# 02 - Prepare Dataset

This notebook loads the raw Firestore CSV, cleans invalid readings, creates ML features, creates a `cleanliness_label`, and saves the processed dataset.

In [1]:
from pathlib import Path
import sys
import pandas as pd

cwd = Path.cwd()
if cwd.name == "notebooks":
    BASE_DIR = cwd.parent
elif (cwd / "model building").exists():
    BASE_DIR = cwd / "model building"
else:
    BASE_DIR = cwd

sys.path.append(str(BASE_DIR / "src"))
RAW_PATH = BASE_DIR / "data" / "raw" / "firestore_sensor_readings_raw.csv"
PROCESSED_PATH = BASE_DIR / "data" / "processed" / "cleansight_processed_dataset.csv"
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)

print("Raw path:", RAW_PATH)
print("Processed path:", PROCESSED_PATH)

Raw path: /Users/vinushan/Documents/VAUED-Project/model building/data/raw/firestore_sensor_readings_raw.csv
Processed path: /Users/vinushan/Documents/VAUED-Project/model building/data/processed/cleansight_processed_dataset.csv


## Load raw data

This file is created by notebook 01. It contains one row per sensor reading with session metadata attached.

In [2]:
raw_df = pd.read_csv(RAW_PATH)
print("Raw rows:", len(raw_df))
raw_df.head()

Raw rows: 1171


,session_id,house_id,room_id,session_type,device_id,start_time,end_time,status,reading_interval_seconds,total_readings,...,reading_id,timestamp_ms,recorded_at,dust,air_quality,temperature,humidity,dust_level,sensor_status,notes
0,session_155a4e33a1e5,Frenten,Bed,after,esp32_all_sensors_01,2026-04-23T08:45:17.411000+00:00,2026-04-23T08:46:24.906000+00:00,completed,4,15,...,reading_1a36383aaf1f,1776933925347,2026-04-23T08:45:28.150000+00:00,161.80,255.41,28.42,69.21,heavy,ok,NaN
1,session_155a4e33a1e5,Frenten,Bed,after,esp32_all_sensors_01,2026-04-23T08:45:17.411000+00:00,2026-04-23T08:46:24.906000+00:00,completed,4,15,...,reading_1c0493cabf82,1776933941350,2026-04-23T08:45:42.253000+00:00,77.22,142.90,28.61,68.14,slight,ok,NaN
2,session_155a4e33a1e5,Frenten,Bed,after,esp32_all_sensors_01,2026-04-23T08:45:17.411000+00:00,2026-04-23T08:46:24.906000+00:00,completed,4,15,...,reading_296f7b5ef1f1,1776933937349,2026-04-23T08:45:40.531000+00:00,103.80,167.56,28.53,66.58,heavy,ok,NaN
3,session_155a4e33a1e5,Frenten,Bed,after,esp32_all_sensors_01,2026-04-23T08:45:17.411000+00:00,2026-04-23T08:46:24.906000+00:00,completed,4,15,...,reading_3b72f9cd8ad1,1776933982285,2026-04-23T08:46:22.961000+00:00,48.60,101.64,28.00,64.06,slight,ok,NaN
4,session_155a4e33a1e5,Frenten,Bed,after,esp32_all_sensors_01,2026-04-23T08:45:17.411000+00:00,2026-04-23T08:46:24.906000+00:00,completed,4,15,...,reading_4e63c77ae9c4,1776933933345,2026-04-23T08:45:35.101000+00:00,131.52,208.98,28.61,67.09,heavy,ok,NaN


## Clean the dataset

Cleaning steps:

- Remove rows missing `dust`, `air_quality`, `temperature`, `humidity`, or `timestamp_ms`.
- Keep only `sensor_status = OK` readings when sensor status exists.
- Convert `timestamp_ms` into a datetime column.
- Sort readings by session and time.
- Remove duplicate readings.
- Remove impossible sensor values such as negative dust, negative air quality, unrealistic indoor temperature, or humidity outside 0 to 100.

In [3]:
from preprocessing import clean_dataset

clean_df = clean_dataset(raw_df)
print("Rows after cleaning:", len(clean_df))
clean_df.head()

Rows after cleaning: 1171


,session_id,house_id,room_id,session_type,device_id,start_time,end_time,status,reading_interval_seconds,total_readings,...,timestamp_ms,recorded_at,dust,air_quality,temperature,humidity,dust_level,sensor_status,notes,timestamp
0,session_155a4e33a1e5,Frenten,Bed,after,esp32_all_sensors_01,2026-04-23T08:45:17.411000+00:00,2026-04-23T08:46:24.906000+00:00,completed,4,15,...,1776933917345,2026-04-23 08:45:20.475000+00:00,218.74,328.04,28.59,66.28,heavy,ok,NaN,2026-04-23 08:45:17.345
1,session_155a4e33a1e5,Frenten,Bed,after,esp32_all_sensors_01,2026-04-23T08:45:17.411000+00:00,2026-04-23T08:46:24.906000+00:00,completed,4,15,...,1776933921349,2026-04-23 08:45:25.113000+00:00,191.38,296.62,28.62,69.23,heavy,ok,NaN,2026-04-23 08:45:21.349
2,session_155a4e33a1e5,Frenten,Bed,after,esp32_all_sensors_01,2026-04-23T08:45:17.411000+00:00,2026-04-23T08:46:24.906000+00:00,completed,4,15,...,1776933925347,2026-04-23 08:45:28.150000+00:00,161.80,255.41,28.42,69.21,heavy,ok,NaN,2026-04-23 08:45:25.347
3,session_155a4e33a1e5,Frenten,Bed,after,esp32_all_sensors_01,2026-04-23T08:45:17.411000+00:00,2026-04-23T08:46:24.906000+00:00,completed,4,15,...,1776933929349,2026-04-23 08:45:30.825000+00:00,136.90,212.09,28.46,70.83,heavy,ok,NaN,2026-04-23 08:45:29.349
4,session_155a4e33a1e5,Frenten,Bed,after,esp32_all_sensors_01,2026-04-23T08:45:17.411000+00:00,2026-04-23T08:46:24.906000+00:00,completed,4,15,...,1776933933345,2026-04-23 08:45:35.101000+00:00,131.52,208.98,28.61,67.09,heavy,ok,NaN,2026-04-23 08:45:33.345


## Feature engineering

The pipeline creates time, rolling mean, and lag features:

- `hour_of_day`
- `minute_of_hour`
- rolling means over the latest 3 readings
- previous-reading lag values
- `next_dust` for the dust forecasting model

In [4]:
from feature_engineering import build_features

processed_df = build_features(clean_df)
print("Processed rows:", len(processed_df))
processed_df.head()

Processed rows: 1171


,session_id,house_id,room_id,session_type,device_id,start_time,end_time,status,reading_interval_seconds,total_readings,...,air_quality_rolling_mean_3,temperature_rolling_mean_3,humidity_rolling_mean_3,dust_lag_1,air_quality_lag_1,temperature_lag_1,humidity_lag_1,next_dust,cleanliness_label,label_source
0,session_155a4e33a1e5,Frenten,Bed,after,esp32_all_sensors_01,2026-04-23T08:45:17.411000+00:00,2026-04-23T08:46:24.906000+00:00,completed,4,15,...,328.040000,28.590000,66.280000,218.74,328.04,28.59,66.28,191.38,dirty,weak_sensor_rule
1,session_155a4e33a1e5,Frenten,Bed,after,esp32_all_sensors_01,2026-04-23T08:45:17.411000+00:00,2026-04-23T08:46:24.906000+00:00,completed,4,15,...,312.330000,28.605000,67.755000,218.74,328.04,28.59,66.28,161.80,dirty,weak_sensor_rule
2,session_155a4e33a1e5,Frenten,Bed,after,esp32_all_sensors_01,2026-04-23T08:45:17.411000+00:00,2026-04-23T08:46:24.906000+00:00,completed,4,15,...,293.356667,28.543333,68.240000,191.38,296.62,28.62,69.23,136.90,dirty,weak_sensor_rule
3,session_155a4e33a1e5,Frenten,Bed,after,esp32_all_sensors_01,2026-04-23T08:45:17.411000+00:00,2026-04-23T08:46:24.906000+00:00,completed,4,15,...,254.706667,28.500000,69.756667,161.80,255.41,28.42,69.21,131.52,needs_attention,weak_sensor_rule
4,session_155a4e33a1e5,Frenten,Bed,after,esp32_all_sensors_01,2026-04-23T08:45:17.411000+00:00,2026-04-23T08:46:24.906000+00:00,completed,4,15,...,225.493333,28.496667,69.043333,136.90,212.09,28.46,70.83,103.80,needs_attention,weak_sensor_rule


## Label creation note

`cleanliness_label` is created using this priority:

1. Use manual labels if they exist in `cleanliness_label`, `manual_label`, `label`, or readable `notes`.
2. Keep `session_type` for analysis, but do not treat it as a direct cleanliness label by default.
3. If no manual label exists, create temporary weak labels using dust and air quality.

These weak labels are only for the first ML pipeline and demo. They should be improved later with real manual cleaning observations.

In [5]:
print(processed_df["cleanliness_label"].value_counts(dropna=False))
print(processed_df["label_source"].value_counts(dropna=False))

cleanliness_label
dirty              412
needs_attention    412
clean              347
Name: count, dtype: int64
label_source
weak_sensor_rule    1171
Name: count, dtype: int64


## Save processed dataset

This CSV is the input for model training and evaluation.

In [6]:
processed_df.to_csv(PROCESSED_PATH, index=False)
print("Saved processed dataset to:", PROCESSED_PATH)

Saved processed dataset to: /Users/vinushan/Documents/VAUED-Project/model building/data/processed/cleansight_processed_dataset.csv
